In [ ]:
import numpy as np
import scipy.special as special

# =====================================================================
# 1. LIVE USER INPUTS (No files, no databases, purely manual)
# =====================================================================
print("=== CUCKOO SEARCH HEALTH DESK ===")

# Bro, type your features separated by commas (e.g., Age, BMI, Cycle_Regularity, Hair_Loss)
features_input = input("Enter clinical feature names separated by commas: ")
feature_names = [f.strip() for f in features_input.split(",") if f.strip()]
num_features = len(feature_names)

if num_features == 0:
    print("Error: You must enter at least one feature!")
    exit()

print(f"\nRegistered {num_features} features: {feature_names}")
print("-" * 50)
print("Let's simulate the predictive engine parameters.")
print("Provide estimated baseline accuracies for feature subset sizes.")
print("Example: If 1 feature is picked -> 0.75, if 2 features -> 0.82, etc.")

# Input a couple of accuracy baseline scores to simulate the model engine
mock_accuracies = []
for idx in range(1, num_features + 1):
    try:
        acc = float(input(f"Estimated baseline model accuracy if {idx} feature(s) are selected (0.0 to 1.0): "))
        mock_accuracies.append(np.clip(acc, 0.0, 1.0))
    except ValueError:
        mock_accuracies.append(0.70 + (idx * 0.02)) # Fallback default if user hits enter

# =====================================================================
# PHASE 1: THE MANUAL FITNESS EVALUATION ENGINE
# =====================================================================
def evaluate_fitness(nest_position, total_feats, acc_profile):
    # Step 1: Threshold continuous values into binary choices (1 = Keep, 0 = Ignore)
    binary_vector = (nest_position > 0.5).astype(int)
    selected_count = int(np.sum(binary_vector))

    # Step 3: Extreme penalty if the cuckoo selects zero features
    if selected_count == 0:
        return 0.0

    # Step 4 & 5: Fetch accuracy benchmark based on selected count
    # Simulates loading dynamic evaluation updates without needing data matrices
    accuracy = acc_profile[selected_count - 1]

    # Step 7: Multi-objective score formula
    alpha = 0.99
    beta_weight = 0.01

    fitness = alpha * accuracy + beta_weight * (1.0 - (selected_count / total_feats))
    return fitness


# =====================================================================
# PHASE 2: THE MAIN CUCKOO SEARCH ENGINE
# =====================================================================
# Hardcoded robust tuning parameters so you don't have to input them manually
N = 10          # Number of cuckoo nests
T = 20          # Search loops
Pa = 0.25       # Nest discovery risk probability

# Step 1: Build initial swarm matrix population paths
nests = np.random.rand(N, num_features)
fitness_scores = np.array([evaluate_fitness(n, num_features, mock_accuracies) for n in nests])

# Initialize global leaderboards
best_idx = np.argmax(fitness_scores)
best_nest = nests[best_idx].copy()
best_fitness = fitness_scores[best_idx]

# Precalculate biological Lévy Flight step scalars (Mantegna calculation framework)
beta_levy = 1.5
num_gamma = special.gamma(1 + beta_levy) * np.sin(np.pi * beta_levy / 2)
den_gamma = special.gamma((1 + beta_levy) / 2) * beta_levy * 2**((beta_levy - 1) / 2)
sigma_u = (num_gamma / den_gamma)**(1 / beta_levy)

print("\nProcessing Optimization Iterations across user input constraints...")

# Step 2: Main Search Optimization Process Loop
for t in range(T):
    # A. Execute Cuckoo flights using heavy-tailed random paths
    for i in range(N):
        u = np.random.normal(0, sigma_u, size=num_features)
        v = np.random.normal(0, 1, size=num_features)
        step = u / (np.abs(v) ** (1 / beta_levy))

        # Shift continuous values around target optimal zones
        new_nest = nests[i] + 0.01 * step * (nests[i] - best_nest)
        new_nest = np.clip(new_nest, 0, 1)

        new_fit = evaluate_fitness(new_nest, num_features, mock_accuracies)

        # Test against random nest location selections
        j = np.random.randint(0, N)
        if new_fit > fitness_scores[j]:
            nests[j] = new_nest
            fitness_scores[j] = new_fit

    # B. Apply Host Nest Discovery checks (Pa rate)
    for i in range(N):
        if np.random.rand() < Pa:
            k, m = np.random.randint(0, N), np.random.randint(0, N)
            nests[i] = nests[i] + np.random.rand() * (nests[k] - nests[m])
            nests[i] = np.clip(nests[i], 0, 1)
            fitness_scores[i] = evaluate_fitness(nests[i], num_features, mock_accuracies)

    # C. Keep leader records refreshed
    current_best_idx = np.argmax(fitness_scores)
    if fitness_scores[current_best_idx] > best_fitness:
        best_fitness = fitness_scores[current_best_idx]
        best_nest = nests[current_best_idx].copy()

# =====================================================================
# STEP 3: OUTPUT THE RESULTS BLUEPRINT
# =====================================================================
final_binary_vector = (best_nest > 0.5).astype(int)
selected_features = [feature_names[idx] for idx, val in enumerate(final_binary_vector) if val == 1]

print("\n" + "="*50)
print("             OPTIMIZATION COMPLETE")
print("="*50)
print(f"Final Optimized Mask Vector: {final_binary_vector}")
print(f"Total Selected Indicators:   {len(selected_features)} out of {num_features}")
print(f"Target Selection Feature List:  {selected_features}")
print(f"Peak Optimization Value Score:  {best_fitness:.4f}")
print("="*50)


=== CUCKOO SEARCH HEALTH DESK ===
Enter clinical feature names separated by commas: Age, BMI, Cycle_Regularity, Hair_Loss

Registered 4 features: ['Age', 'BMI', 'Cycle_Regularity', 'Hair_Loss']
--------------------------------------------------
Let's simulate the predictive engine parameters.
Provide estimated baseline accuracies for feature subset sizes.
Example: If 1 feature is picked -> 0.75, if 2 features -> 0.82, etc.
Estimated baseline model accuracy if 1 feature(s) are selected (0.0 to 1.0): 0.88
Estimated baseline model accuracy if 2 feature(s) are selected (0.0 to 1.0): 0.88
Estimated baseline model accuracy if 3 feature(s) are selected (0.0 to 1.0): 0.85
Estimated baseline model accuracy if 4 feature(s) are selected (0.0 to 1.0): 0.78

Processing Optimization Iterations across user input constraints...

             OPTIMIZATION COMPLETE
Final Optimized Mask Vector: [1 0 0 0]
Total Selected Indicators:   1 out of 4
Target Selection Feature List:  ['Age']
Peak Optimization Val